# 03 · Demand Twin (Step 4 — first digital twin)

Materializes the **Demand / Customer twin**: the canonical current-state and metric
time-series of demand. Everything downstream (inventory targets, MRP, production priority,
distribution) reads demand from here, not from raw sales orders.

**Owns** (do not redefine elsewhere): firm/open demand, backlog, forecast demand,
actual-vs-forecast accuracy/bias, demand volatility, and customer/product context.

**Inputs:** `gold.salesorders` (actual demand + order book), `gold.demandplan` (current
forecast, from notebook 02), `monitoring.demand_forecast_accuracy` (backtest quality),
`gold.materialsmd` / `gold.customersmd` (context).

**Outputs (in `monitoring`):**
- `demand_twin_current` — one row per series (material × plant × channel), latest state.
- `demand_twin_metrics_timeseries` — per series × period: actual demand history + forward
  forecast (a continuous demand timeline).

Both are rebuilt deterministically from source each run (idempotent overwrite). Pure
SQL/DataFrame — serverless-safe (no `.cache()`).

## Config
Set catalog/schema and the source column mapping. Defaults = CoffCoff (`mickmickstableaws_catalog.gold`).

In [0]:
dbutils.widgets.text("source_catalog", "mickmickstableaws_catalog", "Source catalog")
dbutils.widgets.text("source_schema", "gold", "Source schema")
dbutils.widgets.text("target_catalog", "mickmickstableaws_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("granularity", "W", "Demand bucket: D | W | M")
dbutils.widgets.text("demand_date_basis", "ORDER_DATE", "ORDER_DATE | REQUESTED_DELIVERY_DATE")
dbutils.widgets.text("backlog_lookback_days", "182", "Backlog: how far back an overdue open order still counts")

import uuid
from pyspark.sql import functions as F

SRC = f"{dbutils.widgets.get('source_catalog').strip()}.{dbutils.widgets.get('source_schema').strip()}"
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
GRANULARITY = dbutils.widgets.get("granularity").strip().upper()
DEMAND_BASIS = dbutils.widgets.get("demand_date_basis").strip().upper()
BACKLOG_DAYS = int(dbutils.widgets.get("backlog_lookback_days"))
TRUNC = {"D": "day", "W": "week", "M": "month"}[GRANULARITY]
REFRESH_ID = str(uuid.uuid4())

# Source objects
SALES = f"{SRC}.salesorders"
MAT   = f"{SRC}.materialsmd"
CUST  = f"{SRC}.customersmd"
FCST  = f"{SRC}.demandplan"                     # populated by notebook 02 (current version)
ACC   = f"{FQ_MON}.demand_forecast_accuracy"
OD    = f"{SRC}.outbounddeliveries"             # source deliveries, to net the order book

# Output twin tables
CUR_OUT = f"{FQ_MON}.demand_twin_current"
TS_OUT  = f"{FQ_MON}.demand_twin_metrics_timeseries"

# --- salesorders column mapping (EXTEND for your business) ---
COL_MANDT    = "Client_MANDT"
COL_MATERIAL = "MaterialNumber_MATNR"
COL_PLANT    = "Plant_WERKS"
COL_CHANNEL  = "DistributionChannel_VTWEG"
COL_QTY      = "CumulativeOrderQuantity_KWMENG"
COL_UOM      = "SalesUnit_VRKME"
COL_ORDER_DT = "CreationDate_ERDAT"
COL_REQ_DT   = "RequestedDeliveryDate_VDATU"
COL_REJ      = "RejectionReason_ABGRU"          # non-empty => line rejected (exclude)
COL_DOC      = "SalesDocument_VBELN"
COL_CUST     = "SoldToParty_KUNNR"
DEMAND_DT = COL_ORDER_DT if DEMAND_BASIS == "ORDER_DATE" else COL_REQ_DT

print(f"Source: {SALES}\nForecast: {FCST}\nOutputs: {CUR_OUT} , {TS_OUT}\n"
      f"Grain: {GRANULARITY}  Demand basis: {DEMAND_BASIS}")

## 1. Actual demand history (the demand signal)
Aggregate non-rejected sales-order lines into periodic demand per series. This view feeds
both the current-state rollups and the metric time-series.

In [0]:
demand_hist = spark.sql(f"""
SELECT
  `{COL_MANDT}`    AS mandt,
  `{COL_MATERIAL}` AS material,
  `{COL_PLANT}`    AS plant,
  `{COL_CHANNEL}`  AS channel,
  date_trunc('{TRUNC}', `{DEMAND_DT}`) AS period_start,
  SUM(CAST(`{COL_QTY}` AS DOUBLE))     AS actual_demand_qty,
  COUNT(DISTINCT `{COL_DOC}`)          AS order_count,
  MAX(`{COL_UOM}`)                     AS uom
FROM {SALES}
WHERE `{DEMAND_DT}` IS NOT NULL AND `{COL_MATERIAL}` IS NOT NULL
  AND (`{COL_REJ}` IS NULL OR `{COL_REJ}` = '')
GROUP BY 1,2,3,4,5
""")
demand_hist.createOrReplaceTempView("demand_hist")

AS_OF = spark.sql("SELECT max(period_start) m FROM demand_hist").collect()[0]["m"]
print(f"Demand history rows: {demand_hist.count():,}  as_of: {AS_OF}")

## 2. Component inputs
Rolling demand stats & volatility, the forward/overdue order book, the current forecast, its
backtest accuracy, product context, and the top customer per series. Each is a temp view; the
current-state table is assembled from them in step 3.

_`open_demand_qty` / `backlog_qty` are **net of actual deliveries** (joined from the
`gold.outbounddeliveries` source — not the Distribution twin): open = forward unfulfilled
demand, backlog = overdue unfulfilled demand. Delivered qty is assumed in the same sales unit._

In [0]:
# Rolling demand stats + volatility (windows relative to AS_OF).
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW stats_v AS
SELECT mandt, material, plant, channel, MAX(uom) AS uom,
  AVG(CASE WHEN period_start > date_sub(DATE('{AS_OF}'), 28)  THEN actual_demand_qty END) AS demand_4w_avg,
  AVG(CASE WHEN period_start > date_sub(DATE('{AS_OF}'), 91)  THEN actual_demand_qty END) AS demand_13w_avg,
  AVG(CASE WHEN period_start > date_sub(DATE('{AS_OF}'), 364) THEN actual_demand_qty END) AS demand_52w_avg,
  STDDEV_SAMP(CASE WHEN period_start > date_sub(DATE('{AS_OF}'), 364) THEN actual_demand_qty END) AS demand_52w_std,
  AVG(CASE WHEN period_start <= date_sub(DATE('{AS_OF}'), 91)
            AND period_start >  date_sub(DATE('{AS_OF}'), 182) THEN actual_demand_qty END) AS demand_prev_13w_avg,
  MAX(period_start) AS last_demand_period,
  SUM(actual_demand_qty) AS lifetime_demand_qty
FROM demand_hist GROUP BY 1,2,3,4
""")

# Order book NET of actual deliveries. Delivered qty per sales-order line comes from the
# gold.outbounddeliveries SOURCE (linked via ReferenceSDDocument_VGBEL / ...Item_VGPOS) — we
# read the source table, not the Distribution twin, so there is no twin-to-twin dependency.
# open_demand_qty = forward unfulfilled (requested date in the future);
# backlog_qty      = overdue unfulfilled (requested date passed, within the lookback window).
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW delivered_v AS
SELECT ReferenceSDDocument_VGBEL AS vbeln, ReferenceSDDocumentItem_VGPOS AS posnr,
       SUM(DeliveryQuantity_LFIMG) AS delivered_qty
FROM {OD}
WHERE ReferenceSDDocument_VGBEL IS NOT NULL
GROUP BY 1,2
""")
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW orderbook_v AS
WITH lines AS (
  SELECT so.`{COL_MANDT}` AS mandt, so.`{COL_MATERIAL}` AS material, so.`{COL_PLANT}` AS plant,
         so.`{COL_CHANNEL}` AS channel, so.`{COL_REQ_DT}` AS req_dt, so.`{COL_ORDER_DT}` AS order_dt,
         GREATEST(CAST(so.`{COL_QTY}` AS DOUBLE) - COALESCE(d.delivered_qty, 0), 0) AS unfulfilled_qty
  FROM {SALES} so
  LEFT JOIN delivered_v d
    ON so.`{COL_DOC}` = d.vbeln AND so.Item_POSNR = d.posnr
  WHERE so.`{COL_MATERIAL}` IS NOT NULL AND (so.`{COL_REJ}` IS NULL OR so.`{COL_REJ}` = '')
)
SELECT mandt, material, plant, channel,
  SUM(CASE WHEN req_dt >= DATE('{AS_OF}') THEN unfulfilled_qty END) AS open_demand_qty,
  SUM(CASE WHEN req_dt <  DATE('{AS_OF}')
            AND order_dt > date_sub(DATE('{AS_OF}'), {BACKLOG_DAYS})
           THEN unfulfilled_qty END) AS backlog_qty
FROM lines GROUP BY 1,2,3,4
""")

# Current forecast (from gold.demandplan, current version only).
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW forecast_v AS
SELECT Client_MANDT AS mandt, MaterialNumber_MATNR AS material, Plant_WERKS AS plant,
       DistributionChannel_VTWEG AS channel,
  MAX(ForecastVersion) AS forecast_version, MAX(ForecastOrigin) AS forecast_origin,
  MAX(ForecastModel) AS forecast_model, MAX(ConfidenceLevel) AS confidence_level,
  SUM(ForecastQuantity) AS horizon_total_forecast_qty,
  MAX(CASE WHEN HorizonStep = 1 THEN ForecastQuantity END) AS next_period_forecast_qty,
  MAX(PeriodStart) AS horizon_end_period
FROM {FCST} WHERE IsCurrentVersion = true
GROUP BY 1,2,3,4
""")

# Backtest accuracy for the current forecast version (keyed material x plant x channel).
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW accuracy_v AS
SELECT material, plant, channel,
  MAX(wmape) AS forecast_wmape, MAX(bias) AS forecast_bias, MAX(pi_coverage) AS forecast_pi_coverage
FROM {ACC}
WHERE forecast_version = (SELECT MAX(forecast_version) FROM {ACC})
GROUP BY 1,2,3
""")

# Product context.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW material_v AS
SELECT Materialnumber_Matnr AS material, MAX(Materialtext_Maktx) AS material_text,
       MAX(Materialgroup_Matkl) AS material_group
FROM {MAT} GROUP BY 1
""")

# Top customer per series over the last 52 weeks (+ name/region).
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW topcust_v AS
WITH c AS (
  SELECT `{COL_MATERIAL}` AS material, `{COL_PLANT}` AS plant, `{COL_CHANNEL}` AS channel,
         `{COL_CUST}` AS customer, SUM(CAST(`{COL_QTY}` AS DOUBLE)) AS q
  FROM {SALES}
  WHERE `{COL_ORDER_DT}` > date_sub(DATE('{AS_OF}'), 364)
    AND (`{COL_REJ}` IS NULL OR `{COL_REJ}` = '') AND `{COL_MATERIAL}` IS NOT NULL
  GROUP BY 1,2,3,4
),
r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY material,plant,channel ORDER BY q DESC) AS rn,
         SUM(q) OVER (PARTITION BY material,plant,channel) AS tot
  FROM c
)
SELECT r.material, r.plant, r.channel, r.customer AS top_customer_id,
       CASE WHEN r.tot > 0 THEN r.q / r.tot END AS top_customer_share,
       MAX(cu.Name1_NAME1) AS top_customer_name, MAX(cu.CustomerRegion_REGIO) AS top_customer_region
FROM r LEFT JOIN {CUST} cu ON r.customer = cu.CustomerNumber_KUNNR
WHERE r.rn = 1
GROUP BY r.material, r.plant, r.channel, r.customer, r.q, r.tot
""")
print("Component views ready: stats_v, orderbook_v, forecast_v, accuracy_v, material_v, topcust_v")

## 3. Build `demand_twin_current`
One row per series with state, forecast, quality, volatility, order book, and context.

In [0]:
current = spark.sql(f"""
SELECT
  s.mandt, s.material, s.plant, s.channel,
  m.material_text, m.material_group, s.uom,
  s.demand_4w_avg, s.demand_13w_avg, s.demand_52w_avg, s.demand_52w_std,
  CASE WHEN s.demand_52w_avg > 0 THEN s.demand_52w_std / s.demand_52w_avg END AS demand_cov_52w,
  CASE WHEN s.demand_prev_13w_avg > 0 THEN s.demand_13w_avg / s.demand_prev_13w_avg END AS demand_trend_13w,
  s.last_demand_period, s.lifetime_demand_qty,
  ob.open_demand_qty, ob.backlog_qty,
  f.next_period_forecast_qty, f.horizon_total_forecast_qty, f.horizon_end_period,
  f.forecast_version, f.forecast_origin, f.forecast_model, f.confidence_level,
  a.forecast_wmape, a.forecast_bias, a.forecast_pi_coverage,
  tc.top_customer_id, tc.top_customer_name, tc.top_customer_region, tc.top_customer_share
FROM stats_v s
LEFT JOIN material_v  m  ON s.material = m.material
LEFT JOIN orderbook_v ob ON s.mandt=ob.mandt AND s.material=ob.material AND s.plant=ob.plant AND s.channel=ob.channel
LEFT JOIN forecast_v  f  ON s.mandt=f.mandt  AND s.material=f.material  AND s.plant=f.plant  AND s.channel=f.channel
LEFT JOIN accuracy_v  a  ON s.material=a.material AND s.plant=a.plant AND s.channel=a.channel
LEFT JOIN topcust_v   tc ON s.material=tc.material AND s.plant=tc.plant AND s.channel=tc.channel
""")

current = (current
    .withColumn("as_of_date", F.lit(AS_OF).cast("date"))
    .withColumn("refresh_id", F.lit(REFRESH_ID))
    .withColumn("generated_at", F.current_timestamp()))

(current.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(CUR_OUT))
spark.sql(f"COMMENT ON TABLE {CUR_OUT} IS 'Demand twin current state per material x plant x channel: demand rate, volatility, order book, current forecast, accuracy, context.'")
print(f"Wrote {current.count():,} series to {CUR_OUT}")

## 4. Build `demand_twin_metrics_timeseries`
A continuous demand timeline per series: historical **ACTUAL** periods + forward **FORECAST**
periods (with bounds). Drives trend/anomaly/plan-vs-actual views.

In [0]:
timeseries = spark.sql(f"""
SELECT mandt, material, plant, channel, period_start, 'ACTUAL' AS period_type,
  actual_demand_qty,
  CAST(NULL AS DOUBLE) AS forecast_qty, CAST(NULL AS DOUBLE) AS forecast_lower_qty,
  CAST(NULL AS DOUBLE) AS forecast_upper_qty, order_count, uom
FROM demand_hist
UNION ALL
SELECT Client_MANDT, MaterialNumber_MATNR, Plant_WERKS, DistributionChannel_VTWEG,
       PeriodStart, 'FORECAST' AS period_type,
  CAST(NULL AS DOUBLE) AS actual_demand_qty,
  CAST(ForecastQuantity AS DOUBLE), CAST(ForecastLowerQuantity AS DOUBLE),
  CAST(ForecastUpperQuantity AS DOUBLE), CAST(NULL AS BIGINT) AS order_count, UoM
FROM {FCST} WHERE IsCurrentVersion = true
""")

timeseries = (timeseries
    .withColumn("refresh_id", F.lit(REFRESH_ID))
    .withColumn("generated_at", F.current_timestamp()))

(timeseries.write.mode("overwrite").option("overwriteSchema", "true")
    .partitionBy("period_type").saveAsTable(TS_OUT))
spark.sql(f"COMMENT ON TABLE {TS_OUT} IS 'Demand twin metric time-series per series x period: ACTUAL demand history + forward FORECAST (with bounds).'")
print(f"Wrote {timeseries.count():,} rows to {TS_OUT}")

## Column descriptions
Apply human-readable descriptions to every output column (persisted on rebuild).

In [0]:
# Column descriptions — re-applied after the overwrite writes so they persist in the
# catalog on every rebuild. EXTEND for your business.
COLUMN_COMMENTS = {
    CUR_OUT: {
        "mandt": "SAP client (Mandant) identifier",
        "material": "Material / SKU number",
        "plant": "Plant the demand series belongs to",
        "channel": "Distribution channel",
        "material_text": "Material description",
        "material_group": "Material group / category",
        "uom": "Sales unit of measure for the series",
        "demand_4w_avg": "Average actual demand per period over the last 4 weeks",
        "demand_13w_avg": "Average actual demand per period over the last 13 weeks",
        "demand_52w_avg": "Average actual demand per period over the last 52 weeks",
        "demand_52w_std": "Std deviation of weekly demand over the last 52 weeks",
        "demand_cov_52w": "Coefficient of variation of demand (std/mean) over 52 weeks; higher = more volatile",
        "demand_trend_13w": "Ratio of last-13-week avg demand to the prior 13 weeks; above 1 = rising",
        "last_demand_period": "Most recent period with actual demand",
        "lifetime_demand_qty": "Total actual demand across all history",
        "open_demand_qty": "Forward unfulfilled demand (ordered minus delivered) with requested delivery date in the future",
        "backlog_qty": "Overdue unfulfilled demand (ordered minus delivered), requested date passed, within the lookback window",
        "next_period_forecast_qty": "Forecast quantity for the next period (horizon step 1)",
        "horizon_total_forecast_qty": "Total forecast quantity across the full horizon",
        "horizon_end_period": "Last period covered by the current forecast",
        "forecast_version": "Identifier of the current demand-forecast version",
        "forecast_origin": "Forecast cutoff (last actuals period the forecast was generated from)",
        "forecast_model": "Forecasting model used (e.g. AutoETS)",
        "confidence_level": "Prediction-interval confidence level as a fraction",
        "forecast_wmape": "Backtest weighted MAPE for this series; lower = more accurate",
        "forecast_bias": "Backtest signed bias; positive = over-forecast",
        "forecast_pi_coverage": "Backtest prediction-interval coverage (share of actuals within the interval)",
        "top_customer_id": "Largest-volume customer for this series over the last 52 weeks",
        "top_customer_name": "Name of the top customer",
        "top_customer_region": "Region of the top customer",
        "top_customer_share": "Share of demand from the top customer over the last 52 weeks",
        "as_of_date": "As-of date of this twin snapshot",
        "refresh_id": "Unique id of the refresh run that produced this row",
        "generated_at": "Timestamp this row was generated",
    },
    TS_OUT: {
        "mandt": "SAP client identifier",
        "material": "Material / SKU number",
        "plant": "Plant",
        "channel": "Distribution channel",
        "period_start": "Start date of the period bucket",
        "period_type": "ACTUAL for historical demand, FORECAST for forward periods",
        "actual_demand_qty": "Actual ordered demand in the period (ACTUAL rows)",
        "forecast_qty": "Forecast demand for the period (FORECAST rows)",
        "forecast_lower_qty": "Lower prediction-interval bound (FORECAST rows)",
        "forecast_upper_qty": "Upper prediction-interval bound (FORECAST rows)",
        "order_count": "Distinct sales orders contributing to the period (ACTUAL rows)",
        "uom": "Unit of measure",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
}
for _tbl, _cols in COLUMN_COMMENTS.items():
    for _c, _desc in _cols.items():
        spark.sql(f"ALTER TABLE {_tbl} ALTER COLUMN {_c} COMMENT '{_desc}'")
print(f"Applied column descriptions to {CUR_OUT} and {TS_OUT}")

## 5. Verify

In [0]:
print("=== demand_twin_current (sample) ===")
display(spark.sql(f"""
SELECT material, plant, channel, material_group, demand_13w_avg, demand_cov_52w,
       open_demand_qty, backlog_qty, next_period_forecast_qty, horizon_total_forecast_qty,
       forecast_wmape, forecast_bias, top_customer_name, top_customer_share
FROM {CUR_OUT} ORDER BY horizon_total_forecast_qty DESC NULLS LAST LIMIT 25
"""))

print("=== metric timeseries: row counts by type ===")
display(spark.sql(f"SELECT period_type, COUNT(*) rows, COUNT(DISTINCT material||plant||channel) series, MIN(period_start) first_p, MAX(period_start) last_p FROM {TS_OUT} GROUP BY period_type"))

## Next
Demand twin is live. Next twin: **Inventory** (`04_inventory_twin`) — it connects demand,
procurement, production, and distribution (days-of-cover, stockout/overstock risk,
projected balances), consuming this twin's demand rate and forecast. See the orchestrator
skill, Step 4.